In [0]:
!pip install openai

In [0]:
!pip install python-dotenv

In [0]:
%pip uninstall -y langchain langchain-core langfuse databricks-langchain

%pip install langchain==0.3.26
%pip install langchain-core==0.3.68
%pip install langfuse==2.60.2
%pip install databricks-langchain==0.5.1

dbutils.library.restartPython()

In [0]:
import time

In [0]:
import langchain
import langfuse

print(langchain.__version__)
print(langfuse.__version__)

In [0]:
dbutils.library.restartPython()

In [0]:
import os

from langfuse import Langfuse

from databricks_langchain import ChatDatabricks

from dotenv import load_dotenv

In [0]:
load_dotenv()

In [0]:
openai_api_key = os.getenv("OPENAI_API_KEY")
langfuse_secret_key = os.getenv("LANGFUSE_SECRET_KEY")
langfuse_public_key = os.getenv("LANGFUSE_PUBLIC_KEY")
langfuse_base_url = os.getenv("LANGFUSE_BASE_URL")
base_url = "https://dbc-a7bf52c7-eee8.cloud.databricks.com/serving-endpoints"
# openai_api_key = os.getenv("OPENAI_API_KEY")
# openai_api_key = os.getenv("OPENAI_API_KEY")
# # Set environment variable
# os.environ["OPENAI_API_KEY"] = openai_api_key


In [0]:
langfuse = Langfuse(
    public_key=langfuse_public_key,
    secret_key=langfuse_secret_key,
    host=langfuse_base_url
)

In [0]:
llm = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct",
    temperature=0.7,
    max_tokens=100
)

In [0]:
import time
from datetime import datetime
prompt = "Tell me a funny joke about astronauts"

created_date = datetime.now().date()



In [0]:
trace = langfuse.trace(
    name="tell-joke-trace",
    user_id="user-001"
)

In [0]:
#Generation Span

generation = trace.generation(
    name="joke-generation",
    model="llama3",
    input=prompt
)


In [0]:

status = "SUCCESS"
error_message = ""
start_time = time.time()
try:
    response = llm.invoke(prompt)
    joke = response.content

except Exception as e:
    status = "FAILED"
    error_message = str(e)
    raise

latency = round(
    time.time() - start_time,
    2
)

estimated_tokens = (
    len(prompt.split())
    + len(joke.split())
)

# ----------------------------------------
# COST ESTIMATION
# ----------------------------------------

estimated_cost = round(
    estimated_tokens * 0.000001,
    6
)

generation.end(
    output=joke
)

In [0]:
langfuse.flush()



In [0]:
time.sleep(10)

In [0]:
print(joke)

In [0]:
from pyspark.sql import Row 
from datetime import datetime
from pyspark.sql.functions import col

In [0]:


logs = [
    Row(
        timestamp=datetime.now(),
        user_id="user-001",
        prompt=prompt,
        response=joke,
        model="llama3",
        status = status,
        error_message = error_message,
        trace_id = trace.id,
        latency=latency,
        token_count=estimated_tokens,
        estimated_cost=estimated_cost,
        created_date = created_date
    )
]

df = spark.createDataFrame(logs)

df = df.withColumn(
    "token_count",
    col("token_count").cast("int"))

df.write.mode("append").saveAsTable(
    "llmops.monitoring.ai_requests"
)

print(joke)